<a href="https://colab.research.google.com/github/satyam-1605/RAG-from-basic-to-advance/blob/main/RAG_Application_using_Langchain_GEMINI_API_and_FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain

In [ ]:
!pip install -q langchain-community

In [ ]:
!pip install -q langchain-google-genai tiktoken rapidocr-onnxruntime

In [ ]:
from google.colab import userdata
GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')

In [ ]:
import os
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS

In [ ]:
with open("/content/state_of_the_union.txt","r", encoding="utf8") as f:
  data = f.read()

In [ ]:
loder=TextLoader('state_of_the_union.txt', encoding="utf8")

In [ ]:
document=loder.load()

In [ ]:
print(document[0].page_content)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)

In [ ]:
text_chunks=text_splitter.split_documents(document)

In [ ]:
text_chunks

In [ ]:
print(text_chunks[3].page_content)

In [ ]:
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [ ]:
!pip install -q faiss-cpu

In [ ]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [ ]:
retriever=vectorstore.as_retriever()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [ ]:
prompt=ChatPromptTemplate.from_template(template)

In [ ]:

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
   model="gemini-2.5-flash"

)


In [ ]:
output_parser=StrOutputParser()

In [ ]:
rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [ ]:
rag_chain.invoke("How is the United States supporting Ukraine economically and militarily?")

In [ ]:
rag_chain.invoke("What action is the U.S. taking to address rising gas prices?")